# 🔬 Автоматическое измерение высоты Au наночастиц на АСМ изображениях

**Пайплайн:**
1. Загрузка и предобработка АСМ данных
2. Построение карты подложки (morphological opening)
3. Детекция seed points (LoG)
4. Сегментация SAM 2
5. Измерение высот с локальным baseline
6. Baseline алгоритм (круговые маски) для сравнения
7. Визуализация и статистика

---

## 0. Установка зависимостей

In [1]:
# Запустить один раз
# !pip install numpy scipy scikit-image matplotlib pandas tqdm
# !pip install torch torchvision
# !pip install git+https://github.com/facebookresearch/sam2.git

# Для чтения АСМ файлов (выбери нужное):
# !pip install pySPM          # Bruker .spm
# !pip install igor            # Asylum .ibw
# !pip install pygwyfile        # Gwyddion .gwy

# Скачать чекпоинт SAM 2:
# !wget https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt

## 1. Импорты и конфигурация

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import pandas as pd
from scipy import ndimage
from scipy.ndimage import binary_dilation, binary_erosion
from scipy.linalg import lstsq
from skimage.feature import blob_log
from skimage.filters import gaussian
from skimage.morphology import disk, opening as morph_opening
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────
#  КОНФИГУРАЦИЯ — меняй здесь
# ─────────────────────────────────────────
CFG = {
    # Путь к файлу
    'file_path': '/home/matsu/AFM-analysis/data/2025/18 February/4.007',      # .spm / .ibw / .gwy / .npy
    'file_format': 'spm',              # 'spm' | 'ibw' | 'gwy' | 'npy'
    
    # Параметры сканирования
    'pixel_size_nm': 1.0,              # нм/пиксель — взять из метаданных!
    
    # Ожидаемые размеры частиц
    'particle_min_nm': 3,              # минимальный диаметр, нм
    'particle_max_nm': 80,             # максимальный диаметр, нм
    'min_height_nm': 0.5,             # порог: ниже = шум
    
    # LoG детектор
    'log_threshold': 0.05,             # чувствительность (0.01–0.2)
    'log_overlap': 0.3,                # допустимое перекрытие блобов
    
    # Morphological opening (карта подложки)
    'opening_radius_px': 40,           # > радиус крупнейшей частицы в пикселях
    
    # Кольцо для baseline
    'ring_outer_px': 7,                # ширина кольца (пикс)
    'ring_inner_erode_px': 2,          # отступ от края маски внутрь
    
    # SAM
    'sam_checkpoint': '../sam2.1_hiera_base_plus.pt',
    'sam_config': '../sam2.1_hiera_b+.yaml',
    'device': 'cuda',                  # 'cuda' или 'cpu'
    
    # Colormap для конвертации Z → RGB
    'colormap': 'afmhot',              # попробуй: 'afmhot', 'viridis', 'gray'
    'clip_percentile': 99,             # обрезать выбросы (агрегаты)
}

print('✅ Конфигурация загружена')
print(f"   Файл: {CFG['file_path']}")
print(f"   Размер пикселя: {CFG['pixel_size_nm']} нм")
print(f"   Ожидаемые частицы: {CFG['particle_min_nm']}–{CFG['particle_max_nm']} нм")

✅ Конфигурация загружена
   Файл: /home/matsu/AFM-analysis/data/2025/18 February/4.007
   Размер пикселя: 1.0 нм
   Ожидаемые частицы: 3–80 нм


## 2. Загрузка АСМ данных

In [4]:
def load_afm(file_path: str, fmt: str) -> np.ndarray:
    """
    Загрузка АСМ файла → Z-карта в нанометрах (float32).
    Поддерживает: .spm (Bruker), .ibw (Asylum), .gwy (Gwyddion), .npy
    """
    if fmt == 'npy':
        # Для тестов: numpy массив
        z = np.load(file_path).astype(np.float32)
    
    elif fmt == 'spm':
        import pySPM
        scan = pySPM.Bruker(file_path)
        z = scan.get_channel('Height Sensor').pixels.astype(np.float32)
    
    elif fmt == 'ibw':
        import igor.binarywave as bw
        data = bw.load(file_path)
        z = (data['wave']['wData'][:, :, 0] * 1e9).astype(np.float32)  # м → нм
    
    elif fmt == 'gwy':
        import pygwyfile
        gwy = pygwyfile.load(file_path)
        z = (pygwyfile.util.get_datafield(gwy, '/0/data') * 1e9).astype(np.float32)
    
    else:
        raise ValueError(f'Неизвестный формат: {fmt}')
    
    print(f'✅ Загружено: {z.shape[1]}×{z.shape[0]} пикс, '
          f'Z: [{z.min():.2f}, {z.max():.2f}] нм')
    return z


def make_synthetic_afm(size=256, n_particles=40, seed=42) -> np.ndarray:
    """
    Синтетическое АСМ изображение для тестирования.
    Gaussian-частицы разных размеров + наклон + шум.
    """
    rng = np.random.default_rng(seed)
    z = np.zeros((size, size), dtype=np.float32)
    
    # Наклон подложки
    x, y = np.meshgrid(np.arange(size), np.arange(size))
    z += 0.05 * x + 0.03 * y
    
    # Частицы (гауссовы холмики)
    true_heights = []
    for _ in range(n_particles):
        cy, cx = rng.integers(20, size-20, 2)
        height = rng.uniform(2, 25)          # нм
        radius = rng.uniform(3, 12)          # пикс (~sigma)
        z += height * np.exp(-((x-cx)**2 + (y-cy)**2) / (2*radius**2))
        true_heights.append(height)
    
    # Шум подложки
    z += rng.normal(0, 0.15, z.shape)
    
    print(f'✅ Синтетическое изображение: {n_particles} частиц, '
          f'высоты {np.min(true_heights):.1f}–{np.max(true_heights):.1f} нм')
    return z.astype(np.float32)


# ─── Загружаем данные ─────────────────────
USE_SYNTHETIC = False  # ← False когда есть реальный файл

if USE_SYNTHETIC:
    z_raw = make_synthetic_afm(size=256, n_particles=40)
else:
    z_raw = load_afm(CFG['file_path'], CFG['file_format'])

# Быстрый просмотр
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(z_raw, cmap='afmhot', origin='upper')
plt.colorbar(im, ax=ax, label='Z, нм')
ax.set_title('Исходное АСМ изображение')
ax.set_xlabel('X, пикс')
ax.set_ylabel('Y, пикс')
plt.tight_layout()
plt.show()

Exception: Channel Height Sensor not found

## 3. Предобработка

In [ ]:
def flatten_plane(z: np.ndarray) -> np.ndarray:
    """Вычитание наклонной плоскости методом МНК."""
    h, w = z.shape
    xi, yi = np.meshgrid(np.arange(w), np.arange(h))
    A = np.c_[xi.ravel(), yi.ravel(), np.ones(xi.size)]
    coeffs, *_ = lstsq(A, z.ravel())
    plane = (coeffs[0]*xi + coeffs[1]*yi + coeffs[2]).reshape(h, w)
    return z - plane


def flatten_lines(z: np.ndarray, poly_order: int = 1) -> np.ndarray:
    """
    Построчное выравнивание: убирает горизонтальные строчные артефакты.
    poly_order=1 — линейный, 2 — квадратичный.
    """
    result = np.empty_like(z)
    xi = np.arange(z.shape[1])
    for i, row in enumerate(z):
        coeffs = np.polyfit(xi, row, poly_order)
        result[i] = row - np.polyval(coeffs, xi)
    return result


# ─── Применяем ────────────────────────────
z_flat = flatten_plane(z_raw)
z_flat = flatten_lines(z_flat, poly_order=1)

# Визуализация: до и после
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, z, title in zip(axes, [z_raw, z_flat], ['До выравнивания', 'После выравнивания']):
    im = ax.imshow(z, cmap='afmhot', origin='upper')
    plt.colorbar(im, ax=ax, label='Z, нм')
    ax.set_title(title)
plt.tight_layout()
plt.show()

print(f'Z после выравнивания: [{z_flat.min():.3f}, {z_flat.max():.3f}] нм')

## 4. Карта подложки (morphological opening)

In [ ]:
def get_substrate_map(z: np.ndarray, radius_px: int) -> np.ndarray:
    """
    Morphological opening: структурный элемент (диск) крупнее частиц
    "проходит под" ними и оставляет только подложку.
    
    radius_px должен быть БОЛЬШЕ радиуса самой крупной частицы в пикселях.
    """
    selem = disk(radius_px)
    substrate = morph_opening(z, selem)
    return substrate


radius_px = CFG['opening_radius_px']
substrate = get_substrate_map(z_flat, radius_px)
z_above = z_flat - substrate          # «рельеф» частиц над подложкой

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, title, cmap in zip(
    axes,
    [z_flat, substrate, z_above],
    ['Выровненная Z-карта', 'Карта подложки', 'Частицы (Z − подложка)'],
    ['afmhot', 'afmhot', 'hot']
):
    im = ax.imshow(data, cmap=cmap, origin='upper')
    plt.colorbar(im, ax=ax, label='Z, нм')
    ax.set_title(title)

plt.suptitle(f'Morphological opening, radius={radius_px} пикс', fontsize=12)
plt.tight_layout()
plt.show()

print(f'✅ Карта подложки готова. Совет: если частицы "срезаны" — уменьши opening_radius_px')

## 5. Детекция seed points (LoG)

In [ ]:
def detect_seeds(z_above: np.ndarray, cfg: dict) -> np.ndarray:
    """
    Laplacian of Gaussian (LoG) на карте частиц.
    Возвращает массив [y, x, sigma] для каждого найденного блоба.
    """
    px = cfg['pixel_size_nm']
    
    min_sigma = cfg['particle_min_nm']  / (2 * np.sqrt(2) * px)
    max_sigma = cfg['particle_max_nm']  / (2 * np.sqrt(2) * px)
    
    # Нормализуем для LoG
    z_norm = z_above / z_above.max()
    
    blobs = blob_log(
        z_norm,
        min_sigma=min_sigma,
        max_sigma=max_sigma,
        num_sigma=15,
        threshold=cfg['log_threshold'],
        overlap=cfg['log_overlap'],
    )
    
    return blobs  # [y, x, sigma]


blobs = detect_seeds(z_above, CFG)
print(f'✅ Найдено кандидатов: {len(blobs)}')

# Визуализация seed points
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(z_above, cmap='hot', origin='upper')

for blob in blobs:
    y, x, sigma = blob
    r = sigma * np.sqrt(2)
    circle = plt.Circle((x, y), r, color='cyan', fill=False, linewidth=1.2)
    ax.add_patch(circle)
    ax.plot(x, y, '+', color='cyan', markersize=4)

ax.set_title(f'LoG seed points: {len(blobs)} частиц')
plt.tight_layout()
plt.show()

# Подсказка по настройке
print('\n💡 Слишком много ложных срабатываний → увеличь log_threshold')
print('   Частицы пропускаются → уменьши log_threshold')

## 6. Baseline алгоритм (круговые маски)

In [ ]:
def create_circular_mask(shape, cy, cx, radius):
    """Булева маска — диск радиуса radius с центром (cy, cx)."""
    y, x = np.ogrid[:shape[0], :shape[1]]
    return (x - cx)**2 + (y - cy)**2 <= radius**2


def get_ring_baseline(z: np.ndarray, mask: np.ndarray,
                      outer_px: int, inner_erode_px: int) -> float:
    """
    Локальный baseline: медиана Z в кольце вокруг маски частицы.
    
    outer_px       — насколько расширяем маску наружу
    inner_erode_px — отступ от края маски внутрь (убираем краевые пиксели)
    """
    outer = binary_dilation(mask, iterations=outer_px)
    inner = binary_erosion(mask,  iterations=inner_erode_px)
    ring  = outer & ~inner                   # кольцо снаружи частицы
    
    if ring.sum() == 0:
        return float(np.median(z))           # fallback
    return float(np.median(z[ring]))


def measure_height(z: np.ndarray, mask: np.ndarray,
                   outer_px: int = 7, inner_erode_px: int = 2) -> dict:
    """Возвращает высоту частицы и вспомогательные метрики."""
    baseline = get_ring_baseline(z, mask, outer_px, inner_erode_px)
    z_in_mask = z[mask]
    return {
        'height_nm':  float(z_in_mask.max()  - baseline),
        'mean_nm':    float(z_in_mask.mean() - baseline),
        'baseline_nm': baseline,
        'area_px':    int(mask.sum()),
    }


# ─── Baseline измерения ───────────────────
results_baseline = []

for blob in tqdm(blobs, desc='Baseline (круги)'):
    y, x, sigma = blob
    y, x = int(round(y)), int(round(x))
    radius = sigma * np.sqrt(2)
    
    mask = create_circular_mask(z_flat.shape, y, x, radius)
    
    # Граничные частицы — пропускаем
    if mask.sum() < 4:
        continue
    
    metrics = measure_height(
        z_flat, mask,
        outer_px=CFG['ring_outer_px'],
        inner_erode_px=CFG['ring_inner_erode_px']
    )
    
    if metrics['height_nm'] >= CFG['min_height_nm']:
        results_baseline.append({
            'x': x, 'y': y,
            'sigma_px': sigma,
            'diameter_nm': 2 * radius * CFG['pixel_size_nm'],
            'method': 'baseline_circle',
            **metrics
        })

df_baseline = pd.DataFrame(results_baseline)
print(f'\n✅ Baseline: {len(df_baseline)} частиц измерено')
print(df_baseline[['height_nm', 'diameter_nm', 'area_px']].describe().round(2))

## 7. SAM 2 сегментация

In [ ]:
# ─── Конвертация Z → RGB для SAM ──────────
def afm_to_rgb(z: np.ndarray, colormap: str = 'afmhot',
               clip_percentile: float = 99) -> np.ndarray:
    """
    Z-карта (float) → uint8 RGB для SAM.
    clip_percentile: агрегаты не должны "съедать" контраст одиночных частиц.
    """
    lo, hi = z.min(), np.percentile(z, clip_percentile)
    z_clip = np.clip(z, lo, hi)
    z_norm = (z_clip - lo) / (hi - lo + 1e-9)
    
    cmap = plt.get_cmap(colormap)
    rgb  = (cmap(z_norm)[:, :, :3] * 255).astype(np.uint8)
    return rgb


rgb_image = afm_to_rgb(z_flat, CFG['colormap'], CFG['clip_percentile'])

plt.figure(figsize=(5, 5))
plt.imshow(rgb_image)
plt.title(f'АСМ → RGB ({CFG["colormap"]})')
plt.axis('off')
plt.show()
print('✅ RGB изображение для SAM готово')

In [ ]:
# ─── Загрузка SAM 2 ───────────────────────
# Раскомментировать когда установлен SAM 2

SAM_AVAILABLE = False   # ← True когда SAM установлен

if SAM_AVAILABLE:
    import torch
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    
    sam2_model = build_sam2(
        CFG['sam_config'],
        CFG['sam_checkpoint'],
        device=CFG['device']
    )
    predictor = SAM2ImagePredictor(sam2_model)
    
    # Кодируем изображение ОДИН РАЗ
    predictor.set_image(rgb_image)
    print(f'✅ SAM 2 загружен на {CFG["device"]}')
    print(f'   Image embedding готов — теперь можно быстро сегментировать все {len(blobs)} частиц')
else:
    print('⚠️  SAM не установлен — пропускаем, используем только baseline')
    print('   Установи: pip install git+https://github.com/facebookresearch/sam2.git')

In [ ]:
# ─── SAM сегментация всех частиц ──────────
results_sam = []

if SAM_AVAILABLE:
    for blob in tqdm(blobs, desc='SAM 2 segmentation'):
        y, x, sigma = blob
        y_i, x_i = int(round(y)), int(round(x))
        
        # Point prompt: центр частицы из LoG
        point  = np.array([[x_i, y_i]])
        label  = np.array([1])              # 1 = foreground
        
        masks, scores, _ = predictor.predict(
            point_coords=point,
            point_labels=label,
            multimask_output=True,          # 3 варианта маски
        )
        
        best_mask  = masks[np.argmax(scores)]
        best_score = float(np.max(scores))
        
        # Фильтрация по разумному размеру маски
        area          = best_mask.sum()
        expected_area = np.pi * (sigma * np.sqrt(2))**2
        area_ratio    = area / (expected_area + 1e-6)
        
        if not (0.15 < area_ratio < 6.0):
            continue   # маска слишком большая или маленькая — пропускаем
        
        metrics = measure_height(
            z_flat, best_mask,
            outer_px=CFG['ring_outer_px'],
            inner_erode_px=CFG['ring_inner_erode_px']
        )
        
        if metrics['height_nm'] >= CFG['min_height_nm']:
            results_sam.append({
                'x': x_i, 'y': y_i,
                'sigma_px': sigma,
                'diameter_nm': 2 * sigma * np.sqrt(2) * CFG['pixel_size_nm'],
                'sam_confidence': best_score,
                'area_ratio': area_ratio,
                'method': 'sam2',
                'mask': best_mask,
                **metrics
            })
    
    df_sam = pd.DataFrame([{k:v for k,v in r.items() if k!='mask'}
                           for r in results_sam])
    print(f'\n✅ SAM 2: {len(df_sam)} частиц измерено')
    print(df_sam[['height_nm', 'diameter_nm', 'sam_confidence']].describe().round(2))

else:
    results_sam = []
    df_sam = pd.DataFrame()
    print('SAM пропущен.')

## 8. Визуализация результатов

In [ ]:
def overlay_masks(rgb_img, sam_results, alpha=0.45):
    """Наложение SAM масок на RGB изображение."""
    overlay = rgb_img.copy().astype(float)
    rng = np.random.default_rng(0)
    for r in sam_results:
        color = rng.integers(80, 255, 3).astype(float)
        for c in range(3):
            overlay[:, :, c][r['mask']] = (
                alpha * color[c] + (1-alpha) * overlay[:, :, c][r['mask']]
            )
    return overlay.clip(0, 255).astype(np.uint8)


n_cols = 3 if SAM_AVAILABLE else 2
fig, axes = plt.subplots(1, n_cols, figsize=(6*n_cols, 5))

# Панель 1: baseline
ax = axes[0]
ax.imshow(z_above, cmap='hot', origin='upper')
for r in results_baseline:
    radius = r['sigma_px'] * np.sqrt(2)
    c = plt.Circle((r['x'], r['y']), radius, color='cyan', fill=False, lw=1)
    ax.add_patch(c)
ax.set_title(f'Baseline: {len(results_baseline)} частиц\n(круговые маски)', fontsize=11)
ax.axis('off')

# Панель 2: SAM (если доступен)
if SAM_AVAILABLE and results_sam:
    ax = axes[1]
    overlay = overlay_masks(rgb_image, results_sam)
    ax.imshow(overlay)
    for r in results_sam:
        ax.plot(r['x'], r['y'], 'w+', markersize=5, markeredgewidth=1)
    ax.set_title(f'SAM 2: {len(results_sam)} частиц\n(точные маски)', fontsize=11)
    ax.axis('off')

# Панель 3: гистограмма высот
ax = axes[-1]
bins = np.linspace(0, max(df_baseline['height_nm'].max(), 1) * 1.1, 25)

ax.hist(df_baseline['height_nm'], bins=bins, alpha=0.7,
        color='steelblue', edgecolor='white', label=f'Baseline (n={len(df_baseline)})')

if SAM_AVAILABLE and len(df_sam) > 0:
    ax.hist(df_sam['height_nm'], bins=bins, alpha=0.7,
            color='gold', edgecolor='white', label=f'SAM 2 (n={len(df_sam)})')

ax.set_xlabel('Высота, нм', fontsize=11)
ax.set_ylabel('Количество частиц', fontsize=11)
ax.set_title('Распределение высот', fontsize=11)
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Результаты измерения высот Au наночастиц', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('afm_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ График сохранён: afm_results.png')

## 9. Статистика и сравнение методов

In [ ]:
def print_stats(df: pd.DataFrame, method: str):
    print(f'\n── {method} ──────────────────────────')
    print(f'  Частиц измерено : {len(df)}')
    print(f'  Высота, нм      : {df.height_nm.mean():.2f} ± {df.height_nm.std():.2f}')
    print(f'  Медиана, нм     : {df.height_nm.median():.2f}')
    print(f'  Диапазон, нм    : [{df.height_nm.min():.2f}, {df.height_nm.max():.2f}]')

print_stats(df_baseline, 'Baseline (круговые маски)')

if SAM_AVAILABLE and len(df_sam) > 0:
    print_stats(df_sam, 'SAM 2')
    
    # Сравнение: если обе модели нашли одну и ту же частицу
    print('\n── Сравнение по совпадающим частицам ────')
    
    from scipy.spatial import cKDTree
    coords_b = df_baseline[['x','y']].values
    coords_s = df_sam[['x','y']].values
    tree = cKDTree(coords_b)
    dists, idxs = tree.query(coords_s, k=1)
    
    match_mask = dists < 10   # порог совпадения — 10 пикс
    n_matched = match_mask.sum()
    print(f'  Совпало частиц: {n_matched}')
    
    if n_matched > 0:
        h_b = df_baseline.iloc[idxs[match_mask]]['height_nm'].values
        h_s = df_sam[match_mask]['height_nm'].values
        diff = h_s - h_b
        print(f'  Δ высота (SAM − baseline): {diff.mean():.3f} ± {diff.std():.3f} нм')
        
        from scipy.stats import pearsonr
        r, p = pearsonr(h_b, h_s)
        print(f'  Корреляция Пирсона: r={r:.3f}, p={p:.4f}')

## 10. Экспорт результатов

In [ ]:
# Объединяем всё в один CSV
frames = [df_baseline]
if SAM_AVAILABLE and len(df_sam) > 0:
    frames.append(df_sam)

df_all = pd.concat(frames, ignore_index=True)

# Добавляем физические координаты (нм)
df_all['x_nm'] = df_all['x'] * CFG['pixel_size_nm']
df_all['y_nm'] = df_all['y'] * CFG['pixel_size_nm']

output_cols = ['method', 'x_nm', 'y_nm', 'height_nm', 'mean_nm',
               'diameter_nm', 'area_px', 'baseline_nm']
if SAM_AVAILABLE:
    output_cols += ['sam_confidence']

df_all[output_cols].to_csv('afm_heights.csv', index=False, float_format='%.4f')
print('✅ Результаты сохранены: afm_heights.csv')
df_all[output_cols].head(10)

---
## Следующие шаги

| Шаг | Задача |
|-----|--------|
| 🔧 | Установить SAM 2 и запустить на реальных данных |
| 🎨 | Попробовать разные colormap для конвертации Z→RGB (`viridis`, `gray`, `plasma`) |
| 📊 | Сравнить с ПЭМ/ДРС данными для валидации |
| 🤖 | Попробовать SAM 3 (текстовый промпт: "gold nanoparticle") |
| 📝 | Оформить результаты сравнения baseline vs SAM |

```python
# Полезные параметры для подбора:
# log_threshold     → количество найденных частиц
# opening_radius_px → качество карты подложки  
# colormap          → качество SAM масок
# ring_outer_px     → точность baseline
```